# Built with Meta Llama 3

In [ ]:
!pip install --upgrade transformers bitsandbytes accelerate

In [ ]:
import numpy as np
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import default_data_collator
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import get_linear_schedule_with_warmup

import torch
from torch.utils.data import DataLoader
from datasets import Dataset

import gc
import time

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
df = pd.read_csv('Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv', usecols=["transaction_id","user_id","age","gender","daily_screen_time_hours","social_media_hours","gaming_hours","work_study_hours","sleep_hours","notifications_per_day","app_opens_per_day","weekend_screen_time","stress_level","academic_work_impact","addiction_level","addicted_label"])
df = df.drop(["transaction_id"], axis=1)
print(df)

columns = df.columns.values
print(columns)

In [ ]:
labeled_examples = []
status = ""
pos = 10
neg = 10
index = -1
while len(labeled_examples)!=20:
    index += 1
    sentence= ""
    if int(df.iloc[index]['addicted_label']) == 1:
        if pos == 0:
          continue
        status = "Addicted"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Not Addicted"
        neg -= 1
    sentence += f"User's ID is {df.iloc[index,0]}. "
    sentence += f"Age is {df.iloc[index,1]}. "
    sentence += f"Gender is {df.iloc[index,2]}. "
    sentence += f"Hours on screen daily: {df.iloc[index,3]}/24. "
    sentence += f"Hours on social media daily: {df.iloc[index,4]}/24. "
    sentence += f"Hours gaming daily: {df.iloc[index,5]}/24. "
    sentence += f"Hours working daily: {df.iloc[index,6]}/24. "
    sentence += f"Hours sleeping daily: {df.iloc[index,7]}/24. "
    sentence += f"Notifications per day: {df.iloc[index,8]}. "
    sentence += f"App opens per day: {df.iloc[index,9]}. "
    sentence += f"Hours on phone on a weekend: {df.iloc[index,10]}/24. "
    if df.iloc[index,11] == "Low":
      sentence += f"The user is not stressed. "
    elif df.iloc[index,11] == "Medium":
      sentence += f"The user is stressed. "
    else:
      sentence += f"The user is very stressed. "
    if df.iloc[index,12] == "No":
      sentence += f"The user's work is affected."
    else:
      sentence += f"The user's work is not affected."
    labeled_examples.append((sentence,status))
    df.drop(index, inplace=True)

df = df.reset_index(drop=True)
print(df)
print(labeled_examples)

In [ ]:
data = ""

for k in range(len(labeled_examples)):
  data += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"

print(data)

In [ ]:
schema_context = "Data Columns: [User ID, Age, Gender, Daily Screen Time, Daily Social Media Hours, Daily Gaming Hours, Daily Work Hours, Daily Sleeping Hours, Daily Notifications, Daily App Opens, Weekend Phone Hours, Stress Level, Work Impact, Addicted Label]"
steps = ["Identify all users whose Gender is Male.",
         "Identify all users whose Age is lower than 20.",
         "Find the mean of daily gaming hours of all users."]
'''steps = [
    "Identify all users whose Gender is Female.",
    "Identify all users whose are Addicted.",
    "Identify all users with more than 150 Daily Notifications.",
    "Find the mean Daily Screen Time for all users"
]'''

class Thinker:
  role = "You are a seasoned data scientist named Thinker. Your job is to read the available dataset structure and explain a natural language strategy to achieve the target task. Do not write code and do not try to guess values."

  def generate(self,step,schema_context):
    question_thinker = [
      {"role": "system", "content": self.role},
      {"role": "user", "content": f"\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step}"}
    ]
    answer_thinker = generator(question_thinker, temperature=0.3, max_new_tokens=1024, return_full_text=False)[0]['generated_text'].strip()
    return answer_thinker

class Performer:
  role = "You are a seasoned data scientist named Performer. Your job is to read through the Data Matrix and filter out unneeded rows using the Thinker's Method. EXTREMELY IMPORTANT: You must output the matching matching rows with ALL original column details intact exactly as they appear. Do not summarize into text blocks or lists of IDs.\n\n"

  def generate(self,data,step,answer_thinker,feedback_note):
    performer_content = f"### DATA MATRIX:\n{data}\n\n### STRATEGIC TASK:\n{step}\n\n### METHOD TO IMPLEMENT:\n{answer_thinker}"

    if feedback_note:
            performer_content += f"""### RE-RUN CORRECTION DIRECTIVE: Your previous attempt was rejected by the Auditor. Review the Auditor's step-by-step trace log below, identify the invalid user rows, and remove them from your output while keeping valid rows entirely intact.
                                  ### AUDITOR LOG DETECTED:{feedback_note}"""

    question_performer = [
            {"role": "system", "content": self.role},
            {"role": "user", "content": performer_content}
        ]
    answer_performer = generator(question_performer, temperature=0.1, max_new_tokens=1024, return_full_text=False)[0]['generated_text'].strip()
    return answer_performer

class Evaluator:
  role = f"""SYSTEM INSTRUCTION: You are an adversarial Data Quality Auditor named Evaluator. Your job is to strictly cross-examine the Performer's Results against the baseline data matrix.

        To prevent errors, you MUST process your audit using the following step-by-step verification template format:

        ### AUDIT TRACE LOG:
        - Target Filter Rule: [State the numeric condition or attribute]
        - Row 1 Check: [User ID] has [Value] -> Does this meet the rule? (Yes/No)
        - Row 2 Check: [User ID] has [Value] -> Does this meet the rule? (Yes/No)
        [Continue for all proposed rows]

        ### CONCLUSION:
        If every single checked row evaluated to 'Yes' AND no valid rows from the baseline data were skipped, output exactly: 'VERDICT: SUCCESS'.
        If even one row evaluates to 'No', or if the Performer lazily copy-pasted data from an unrelated step, output exactly: 'VERDICT: FAILED' followed by a description of the hallucination.
        Make sure to evaluate the original data in the same way to see if anything was skipped."""

  def generate(self,data,step,answer_performer,verdict,feedback_note):
    question_evaluator = [
            {"role": "system", "content": self.role},
            {"role": "user", "content": f"### BASELINE DATA: {data} ### TARGET TASK: {step} ### PROPOSED RESULTS TO AUDIT: {answer_performer}"}
    ]
    answer_evaluator = generator(question_evaluator, temperature=0.1, max_new_tokens=1024, return_full_text=False)[0]['generated_text'].strip()

    if "VERDICT: SUCCESS" in answer_evaluator:
      verdict = "Passed"
      print("\n-------- VERDICT: STEP PASSED --------\n")
    else:
      verdict = "Failed"
      feedback_note = answer_evaluator
      print("\n-------- VERDICT: STEP FAILED - RETRYING --------\n")

    return verdict, feedback_note, answer_evaluator

In [ ]:
thinker = Thinker()
performer = Performer()
evaluator = Evaluator()

# **Qwen**

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

In [ ]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
for s in steps:
  answer_performer = ""
  feedback_note = ""
  verdict = "Failed"

  answer_thinker = thinker.generate(s,schema_context)
  print(answer_thinker)

  print("--------------------------------------------------------------------------")

  while verdict != "Passed":
    answer_performer = performer.generate(data,s,answer_thinker,feedback_note)
    print(answer_performer)

    print("--------------------------------------------------------------------------")
    verdict, feedback_note, answer_evaluator = evaluator.generate(data,s,answer_performer,verdict,feedback_note)
    print(answer_evaluator)

  data = answer_performer

# **Mistral**

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
for s in steps:
  answer_performer = ""
  feedback_note = ""
  verdict = "Failed"

  answer_thinker = thinker.generate(s,schema_context)
  print(answer_thinker)

  print("--------------------------------------------------------------------------")

  while verdict != "Passed":
    answer_performer = performer.generate(data,s,answer_thinker,feedback_note)
    print(answer_performer)

    print("--------------------------------------------------------------------------")
    verdict, feedback_note, answer_evaluator = evaluator.generate(data,s,answer_performer,verdict,feedback_note)
    print(answer_evaluator)

  data = answer_performer

# **Mistral + Markdown**

In [ ]:
df_addicted_mini = df[df['addicted_label'] == 1].head(10)
df_nonaddicted_mini = df[df['addicted_label'] == 0].head(10)
merged_df = pd.concat([df_addicted_mini, df_nonaddicted_mini])
drop_indexes = merged_df.index
df_copy = df
df_copy.drop(drop_indexes, inplace=True)
df_copy = df_copy.reset_index(drop=True)

print(df_copy)

merged_df = merged_df.drop(['addiction_level','addicted_label'], axis=1)

table_str = merged_df.to_markdown()

print(table_str)

data = table_str

In [ ]:
schema_context = "Data Columns: [User ID, Age, Gender, Daily Screen Time, Daily Social Media Hours, Daily Gaming Hours, Daily Work Hours, Daily Sleeping Hours, Daily Notifications, Daily App Opens, Weekend Phone Hours, Stress Level, Work Impact, Addicted Label]"
'''steps = ["Identify all users whose Gender is Male.",
         "Identify all users whose Age is lower than 20.",
         "Find the mean of daily gaming hours of all users."]'''
steps = [
    "Identify all users whose Gender is Female.",
    "Identify all users whose are Addicted.",
    "Identify all users with more than 150 Daily Notifications.",
    "Find the mean Daily Screen Time for all users"
]

class Thinker:
  role = "You are a seasoned data scientist named Thinker. Your job is to read the available dataset structure and explain a natural language strategy to achieve the target task. Do not write code and do not try to guess values."

  def generate(self,step,schema_context):
    prompt = f"\n\n###ROLE:\n{self.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step}"
    question_thinker = [
      {"role": "user", "content": prompt}
    ]
    inputs_thinker = tokenizer.apply_chat_template(question_thinker, add_generation_prompt=True, return_tensors="pt").to("cuda")
    prompt_length_thinker = inputs_thinker["input_ids"].size(1)

    with torch.no_grad():
        outputs_thinker = model.generate(**inputs_thinker, max_new_tokens=1024, temperature=0.3, do_sample=True)

    answer_thinker = tokenizer.decode(outputs_thinker[0][prompt_length_thinker:], skip_special_tokens=True).strip()
    return answer_thinker

class Performer:
  role = "You are a seasoned data scientist named Performer. Your job is to read through the Data Matrix and filter out unneeded rows using the Thinker's Method. EXTREMELY IMPORTANT: You must output the matching matching rows with ALL original column details intact exactly as they appear. Do not summarize into text blocks or lists of IDs.\n\n"

  def generate(self,data,step,answer_thinker,feedback_note):
    prompt = f"\n\n###ROLE:\n{self.role}\n\n### DATA MATRIX:\n{data}\n\n### STRATEGIC TASK:\n{step}\n\n### METHOD TO IMPLEMENT:\n{answer_thinker}"

    if feedback_note:
            prompt += f"""### RE-RUN CORRECTION DIRECTIVE: Your previous attempt was rejected by the Auditor. Review the Auditor's step-by-step trace log below, identify the invalid user rows, and remove them from your output while keeping valid rows entirely intact.
                                  ### AUDITOR LOG DETECTED:{feedback_note}"""

    question_performer = [
        {"role": "user", "content": prompt}
        ]
    inputs_performer = tokenizer.apply_chat_template(question_performer, add_generation_prompt=True, return_tensors="pt").to("cuda")
    prompt_length_performer = inputs_performer["input_ids"].size(1)

    with torch.no_grad():
        outputs_performer = model.generate(**inputs_performer, max_new_tokens=1024, temperature=0.1, do_sample=True)

    answer_performer = tokenizer.decode(outputs_performer[0][prompt_length_performer:], skip_special_tokens=True).strip()
    return answer_performer

class Evaluator:
  role = f"""SYSTEM INSTRUCTION: You are an adversarial Data Quality Auditor named Evaluator. Your job is to strictly cross-examine the Performer's Results against the baseline data matrix.

        To prevent errors, you MUST process your audit using the following step-by-step verification template format:

        ### AUDIT TRACE LOG:
        - Target Filter Rule: [State the numeric condition or attribute]
        - Row 1 Check: [User ID] has [Value] -> Does this meet the rule? (Yes/No)
        - Row 2 Check: [User ID] has [Value] -> Does this meet the rule? (Yes/No)
        [Continue for all proposed rows]

        ### CONCLUSION:
        If every single checked row evaluated to 'Yes' AND no valid rows from the baseline data were skipped, output exactly: 'VERDICT: SUCCESS'.
        If even one row evaluates to 'No', or if the Performer lazily copy-pasted data from an unrelated step, output exactly: 'VERDICT: FAILED' followed by a description of the hallucination.
        Make sure to evaluate the original data in the same way to see if anything was skipped."""

  def generate(self,data,step,answer_performer,verdict,feedback_note):
    prompt = f"""###ROLE: {self.role}

        ### BASELINE DATA:
        {data}

        ### TARGET TASK:
        {step}

        ### PROPOSED RESULTS TO AUDIT:
        {answer_performer}"""

    question_evaluator = [
            {"role": "user", "content": prompt}
    ]

    inputs_evaluator = tokenizer.apply_chat_template(question_evaluator, add_generation_prompt=True, return_tensors="pt").to("cuda")
    prompt_length_evaluator = inputs_evaluator["input_ids"].size(1)

    with torch.no_grad():
        outputs_evaluator = model.generate(**inputs_evaluator, max_new_tokens=1024, temperature=0.1, do_sample=True)

    answer_evaluator = tokenizer.decode(outputs_evaluator[0][prompt_length_evaluator:], skip_special_tokens=True).strip()

    if "VERDICT: SUCCESS" in answer_evaluator:
      verdict = "Passed"
      print("\n-------- VERDICT: STEP PASSED --------\n")
    else:
      verdict = "Failed"
      feedback_note = answer_evaluator
      print("\n-------- VERDICT: STEP FAILED - RETRYING --------\n")

    return verdict, feedback_note, answer_evaluator

In [ ]:
thinker = Thinker()
performer = Performer()
evaluator = Evaluator()

In [ ]:
for s in steps:
  answer_performer = ""
  feedback_note = ""
  verdict = "Failed"

  answer_thinker = thinker.generate(s,schema_context)
  print(answer_thinker)

  print("--------------------------------------------------------------------------")

  while verdict != "Passed":
    answer_performer = performer.generate(data,s,answer_thinker,feedback_note)
    print(answer_performer)

    print("--------------------------------------------------------------------------")
    verdict, feedback_note, answer_evaluator = evaluator.generate(data,s,answer_performer,verdict,feedback_note)
    print(answer_evaluator)

  data = answer_performer

# **Llama**

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
schema_context = "Data Columns: [User ID, Age, Gender, Daily Screen Time, Daily Social Media Hours, Daily Gaming Hours, Daily Work Hours, Daily Sleeping Hours, Daily Notifications, Daily App Opens, Weekend Phone Hours, Stress Level, Work Impact, Addicted Label]"
steps = ["Identify all users whose Gender is Male.",
         "Identify all users whose Age is lower than 20.",
         "Find the mean of daily gaming hours of all users."]
'''steps = [
    "Identify all users whose Gender is Female.",
    "Identify all users whose are Addicted.",
    "Identify all users with more than 150 Daily Notifications.",
    "Find the mean Daily Screen Time for all users"
]'''

class Thinker:
  role = "You are a seasoned data scientist named Thinker. Your job is to read the available dataset structure and explain a natural language strategy to achieve the target task. Do not write code and do not try to guess values."

  def generate(self,step,schema_context):
    prompt = f"\n\n###ROLE:\n{self.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step}"
    question_thinker = [
      {"role": "user", "content": prompt}
    ]
    inputs_thinker = tokenizer.apply_chat_template(question_thinker, add_generation_prompt=True, return_tensors="pt").to("cuda")
    prompt_length_thinker = inputs_thinker["input_ids"].size(1)

    with torch.no_grad():
        outputs_thinker = model.generate(**inputs_thinker, max_new_tokens=1024, temperature=0.3, do_sample=True)

    answer_thinker = tokenizer.decode(outputs_thinker[0][prompt_length_thinker:], skip_special_tokens=True).strip()
    return answer_thinker

class Performer:
  role = "You are a seasoned data scientist named Performer. Your job is to read through the Data Matrix and filter out unneeded rows using the Thinker's Method. EXTREMELY IMPORTANT: You must output the matching matching rows with ALL original column details intact exactly as they appear. Do not summarize into text blocks or lists of IDs.\n\n"

  def generate(self,data,step,answer_thinker,feedback_note):
    prompt = f"\n\n###ROLE:\n{self.role}\n\n### DATA MATRIX:\n{data}\n\n### STRATEGIC TASK:\n{step}\n\n### METHOD TO IMPLEMENT:\n{answer_thinker}"

    if feedback_note:
            prompt += f"""### RE-RUN CORRECTION DIRECTIVE: Your previous attempt was rejected by the Auditor. Review the Auditor's step-by-step trace log below, identify the invalid user rows, and remove them from your output while keeping valid rows entirely intact.
                                  ### AUDITOR LOG DETECTED:{feedback_note}"""

    question_performer = [
        {"role": "user", "content": prompt}
        ]
    inputs_performer = tokenizer.apply_chat_template(question_performer, add_generation_prompt=True, return_tensors="pt").to("cuda")
    prompt_length_performer = inputs_performer["input_ids"].size(1)

    with torch.no_grad():
        outputs_performer = model.generate(**inputs_performer, max_new_tokens=1024, temperature=0.1, do_sample=True)

    answer_performer = tokenizer.decode(outputs_performer[0][prompt_length_performer:], skip_special_tokens=True).strip()
    return answer_performer

class Evaluator:
  role = f"""SYSTEM INSTRUCTION: You are an adversarial Data Quality Auditor named Evaluator. Your job is to strictly cross-examine the Performer's Results against the baseline data matrix.

        To prevent errors, you MUST process your audit using the following step-by-step verification template format:

        ### AUDIT TRACE LOG:
        - Target Filter Rule: [State the numeric condition or attribute]
        - Row 1 Check: [User ID] has [Value] -> Does this meet the rule? (Yes/No)
        - Row 2 Check: [User ID] has [Value] -> Does this meet the rule? (Yes/No)
        [Continue for all proposed rows]

        ### CONCLUSION:
        If every single checked row evaluated to 'Yes' AND no valid rows from the baseline data were skipped, output exactly: 'VERDICT: SUCCESS'.
        If even one row evaluates to 'No', or if the Performer lazily copy-pasted data from an unrelated step, output exactly: 'VERDICT: FAILED' followed by a description of the hallucination.
        Make sure to evaluate the original data in the same way to see if anything was skipped."""

  def generate(self,data,step,answer_performer,verdict,feedback_note):
    prompt = f"""###ROLE: {self.role}

        ### BASELINE DATA:
        {data}

        ### TARGET TASK:
        {step}

        ### PROPOSED RESULTS TO AUDIT:
        {answer_performer}"""

    question_evaluator = [
            {"role": "user", "content": prompt}
    ]

    inputs_evaluator = tokenizer.apply_chat_template(question_evaluator, add_generation_prompt=True, return_tensors="pt").to("cuda")
    prompt_length_evaluator = inputs_evaluator["input_ids"].size(1)

    with torch.no_grad():
        outputs_evaluator = model.generate(**inputs_evaluator, max_new_tokens=1024, temperature=0.1, do_sample=True)

    answer_evaluator = tokenizer.decode(outputs_evaluator[0][prompt_length_evaluator:], skip_special_tokens=True).strip()

    if "VERDICT: SUCCESS" in answer_evaluator:
      verdict = "Passed"
      print("\n-------- VERDICT: STEP PASSED --------\n")
    else:
      verdict = "Failed"
      feedback_note = answer_evaluator
      print("\n-------- VERDICT: STEP FAILED - RETRYING --------\n")

    return verdict, feedback_note, answer_evaluator

In [ ]:
thinker = Thinker()
performer = Performer()
evaluator = Evaluator()

In [ ]:
for s in steps:
  answer_performer = ""
  feedback_note = ""
  verdict = "Failed"

  answer_thinker = thinker.generate(s,schema_context)
  print(f"=== THINKER METHOD ===\n{answer_thinker}")

  print("--------------------------------------------------------------------------")

  while verdict != "Passed":
    answer_performer = performer.generate(data,s,answer_thinker,feedback_note)
    print(f"=== PERFORMER RESULT ===\n{answer_performer}")

    print("--------------------------------------------------------------------------")
    verdict, feedback_note, answer_evaluator = evaluator.generate(data,s,answer_performer,verdict,feedback_note)
    print(f"=== EVALUATOR AUDIT ===\n{answer_evaluator}")

  data = answer_performer

In [ ]:
class ComputeProfiler:
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.latencies = []
    self.input_tokens = []
    self.output_tokens = []
    self.peak_vrams = []

  def profile_query(self, query_fn, *args, **kwargs):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    output_text, all_inputs, all_outputs = query_fn(*args, **kwargs)
    end_time = time.perf_counter()

    self.latencies.append(end_time - start_time)
    self.peak_vrams.append(torch.cuda.max_memory_allocated() / (1024**3))

    total_in = sum(len(self.tokenizer.encode(p)) for p in all_inputs)
    total_out = sum(len(self.tokenizer.encode(o)) for o in all_outputs)
    self.input_tokens.append(total_in)
    self.output_tokens.append(total_out)

    return output_text

  def summary(self, model_name: str, method_name: str):
    return {
        "Paradigm / Method": method_name,
        "Model": model_name,
        "Avg Latency (s)": f"{np.mean(self.latencies):.2f} ± {np.std(self.latencies):.2f}",
        "Avg Input Tokens": f"{int(np.mean(self.input_tokens))}",
        "Avg Output Tokens": f"{int(np.mean(self.output_tokens))}",
        "Peak VRAM (GB)": f"{np.max(self.peak_vrams):.2f}",
    }

profiler = ComputeProfiler(tokenizer)

def run_tri_agent_pipeline(step_query, current_data):
  input_prompts = []
  generated_outputs = []

  t_prompt = f"\n\n###ROLE:\n{thinker.role}\n\n### DATASET STRUCTURE:\n{schema_context}\n\n### TARGET TASK:\n{step_query}"
  input_prompts.append(t_prompt)
  answer_thinker = thinker.generate(step_query, schema_context)
  generated_outputs.append(answer_thinker)

  verdict = "Failed"
  feedback_note = ""
  answer_performer = ""
  answer_evaluator = ""
  max_retries = 3

  while verdict != "Passed" and max_retries > 0:
    p_prompt = f"\n\n###ROLE:\n{performer.role}\n\n### DATA MATRIX:\n{current_data}\n\n### STRATEGIC TASK:\n{step_query}\n\n### METHOD TO IMPLEMENT:\n{answer_thinker}"
    if feedback_note:
      p_prompt += f"\n\n### RE-RUN CORRECTION DIRECTIVE: {feedback_note}"
    input_prompts.append(p_prompt)

    answer_performer = performer.generate(current_data, step_query, answer_thinker, feedback_note)
    generated_outputs.append(answer_performer)

    e_prompt = f"###ROLE: {evaluator.role}\n\n### BASELINE DATA:\n{current_data}\n\n### TARGET TASK:\n{step_query}\n\n### PROPOSED RESULTS TO AUDIT:\n{answer_performer}"
    input_prompts.append(e_prompt)

    verdict, feedback_note, answer_evaluator = evaluator.generate(current_data, step_query, answer_performer, verdict, feedback_note)
    generated_outputs.append(answer_evaluator)

    max_retries -= 1

  return answer_performer, input_prompts, generated_outputs

active_data = data
for s in steps:
  print(f"\nProfiling Step: {s}")
  final_result = profiler.profile_query(run_tri_agent_pipeline, s, active_data)
  if final_result and "---" in final_result:
    active_data = final_result

print("\n" + "="*50)
print(profiler.summary("Llama-3-8B-Instruct (4-bit)", "Tri-Agent"))
print("="*50)